In [6]:
# FILTER LABELS
import os
from roboflow import Roboflow

rf = Roboflow(api_key="JK3XRYll8vdLCszAp83e")
project = rf.workspace("adityas-workspace-0eca5").project("badmintoncourtdetectionoffical-b3hl9-dloud")
version = project.version(2)
dataset = version.download("yolov8")    

# ---- Config ----
REMOVE_IDS = [2, 8, 13, 19]
DATASET_PATH = dataset.location

def filter_labels(split):
    labels_dir = os.path.join(DATASET_PATH, split, 'labels')
    
    if not os.path.exists(labels_dir):
        print(f"Skipping {split} — labels dir not found")
        return
    
    processed = 0
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith('.txt'):
            continue
            
        label_path = os.path.join(labels_dir, label_file)
        
        with open(label_path) as f:
            lines = f.readlines()
        
        new_lines = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
                
            parts = line.split()
            cls = parts[0]
            bbox = parts[1:5]          # cx cy w h
            kp_flat = parts[5:]        # all keypoints flat: x y v x y v ...
            
            # Group into triplets (x, y, visibility)
            kp_groups = []
            for i in range(0, len(kp_flat), 3):
                kp_groups.append(kp_flat[i:i+3])
            
            # Keep only keypoints NOT in REMOVE_IDS
            kept_kps = []
            for idx, kp in enumerate(kp_groups):
                if idx not in REMOVE_IDS:
                    kept_kps.extend(kp)
            
            new_line = ' '.join([cls] + bbox + kept_kps)
            new_lines.append(new_line)
        
        # Overwrite with filtered labels
        with open(label_path, 'w') as f:
            f.write('\n'.join(new_lines))
        
        processed += 1
    
    print(f"{split}: processed {processed} label files")

if __name__ == '__main__':
    print(f"Removing keypoint IDs: {REMOVE_IDS}")
    print(f"Keeping 18 keypoints\n")
    
    filter_labels('train')
    filter_labels('valid')
    filter_labels('test')
    
    print("\nDone — labels updated to 18 keypoints")
    print("Remember to update data.yaml kpt_shape to [18, 3]")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to BadmintonCourtDetectionOffical-2 in yolov8:: 100%|██████████| 2393/2393 [00:02<00:00, 911.68it/s] 


Removing keypoint IDs: [2, 8, 13, 19]
Keeping 18 keypoints

train: processed 955 label files
valid: processed 119 label files
test: processed 120 label files

Done — labels updated to 18 keypoints
Remember to update data.yaml kpt_shape to [18, 3]


In [7]:
# Verify the filtering worked
def verify_labels(dataset_path, expected_kpts=18):
    labels_dir = os.path.join(dataset_path, 'train', 'labels')
    issues = []
    
    for label_file in os.listdir(labels_dir)[:20]:  # check first 20
        if not label_file.endswith('.txt'):
            continue
            
        label_path = os.path.join(labels_dir, label_file)
        with open(label_path) as f:
            lines = f.readlines()
        
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            kp_count = (len(parts) - 5) // 3  # subtract cls + bbox, divide by 3
            if kp_count != expected_kpts:
                issues.append(f"{label_file}: expected {expected_kpts} kpts, got {kp_count}")
    
    if issues:
        print("Issues found:")
        for issue in issues:
            print(f"  {issue}")
    else:
        print(f"All checked labels have {expected_kpts} keypoints ✓")

# Call before training
verify_labels(dataset.location)

All checked labels have 18 keypoints ✓


In [ ]:
import os
import shutil
from datetime import datetime
import torch
from ultralytics import YOLO

# ---- Config ----
PROJECT_DIR = 'C:/Users/Aditya/BadmintonML'
RUNS_DIR = f'{PROJECT_DIR}/runs'
RUN_NAME = 'court_keypoints_v2'
WEIGHTS_DIR = f'{RUNS_DIR}/{RUN_NAME}/weights'
BACKUP_DIR = f'{PROJECT_DIR}/backups'

# Point this to your already-downloaded & filtered dataset
DATASET_PATH = 'C:/Users/Aditya/BadmintonML/DataSets/BadmintonCourtDetectionOffical-2'  # update if different
DATA_YAML = os.path.join(DATASET_PATH, 'data.yaml')

def backup_weights(weights_dir, backup_dir):
    os.makedirs(backup_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    for weight_file in ['best.pt', 'last.pt']:
        src = os.path.join(weights_dir, weight_file)
        if os.path.exists(src):
            dst = os.path.join(backup_dir, f"{timestamp}_keypoint_{weight_file}")
            shutil.copy2(src, dst)
            print(f"Backed up: {dst}")

if __name__ == '__main__':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Dataset: {DATASET_PATH}")
    print(f"YAML: {DATA_YAML}\n")

    # Sanity check
    assert os.path.exists(DATA_YAML), f"data.yaml not found at {DATA_YAML}"

    model = YOLO('yolo11s-pose.pt')

    print("Starting keypoint training...")
    try:
        model.train(
            data=DATA_YAML,
            epochs=100,
            imgsz=640,
            batch=8,
            device=0,
            cache='disk',
            optimizer='AdamW',
            lr0=0.0005,
            weight_decay=0.001,
            workers=2,
            patience=25,
            save=True,
            save_period=5,
            plots=True,
            project=RUNS_DIR,
            name=RUN_NAME,
            exist_ok=True,
            amp=True,
            box=7.5,
            cls=0.5,
            pose=12.0,
            kobj=2.0,
            mosaic=0.0,
            mixup=0.0,
            copy_paste=0.0,
            fliplr=0.5,
            scale=0.1,
            hsv_v=0.2,
            hsv_s=0.1,
        )
        print("\nTraining complete")

    except KeyboardInterrupt:
        print("\nInterrupted")

    except Exception as e:
        print(f"\nCrashed: {e}")

    finally:
        backup_weights(WEIGHTS_DIR, BACKUP_DIR)

GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Dataset: C:/Users/Aditya/BadmintonML/BadmintonCourtDetectionOffical-2
YAML: C:/Users/Aditya/BadmintonML/BadmintonCourtDetectionOffical-2\data.yaml

Starting keypoint training...
New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.53  Python-3.13.5 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/Users/Aditya/BadmintonML/BadmintonCourtDetectionOffical-2\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=

In [ ]:
import os
import shutil
from datetime import datetime
from roboflow import Roboflow
from ultralytics import YOLO
import torch

PROJECT_DIR = 'C:/Users/Aditya/BadmintonML'
RUNS_DIR = f'{PROJECT_DIR}/runs'
RUN_NAME = 'court_keypoints_v2'
WEIGHTS_DIR = f'{RUNS_DIR}/{RUN_NAME}/weights'
BACKUP_DIR = f'{PROJECT_DIR}/backups'
REMOVE_IDS = [2, 8, 13, 19]

def backup_weights(weights_dir, backup_dir):
    os.makedirs(backup_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    for weight_file in ['best.pt', 'last.pt']:
        src = os.path.join(weights_dir, weight_file)
        if os.path.exists(src):
            dst = os.path.join(backup_dir, f"{timestamp}_keypoint_{weight_file}")
            shutil.copy2(src, dst)
            print(f"Backed up: {dst}")

def filter_labels(dataset_path, remove_ids):
    print(f"\nFiltering labels — removing keypoint IDs: {remove_ids}")
    
    for split in ['train', 'valid', 'test']:
        labels_dir = os.path.join(dataset_path, split, 'labels')
        if not os.path.exists(labels_dir):
            continue
            
        count = 0
        for label_file in os.listdir(labels_dir):
            if not label_file.endswith('.txt'):
                continue
                
            label_path = os.path.join(labels_dir, label_file)
            with open(label_path) as f:
                lines = f.readlines()
            
            new_lines = []
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                cls = parts[0]
                bbox = parts[1:5]
                kp_flat = parts[5:]
                kp_groups = [kp_flat[i:i+3] for i in range(0, len(kp_flat), 3)]
                
                kept_kps = []
                for idx, kp in enumerate(kp_groups):
                    if idx not in remove_ids:
                        kept_kps.extend(kp)
                
                new_lines.append(' '.join([cls] + bbox + kept_kps))
            
            with open(label_path, 'w') as f:
                f.write('\n'.join(new_lines))
            count += 1
        
        print(f"  {split}: {count} files updated")
    
    print("Filtering complete\n")

def update_yaml(dataset_path):
    yaml_path = os.path.join(dataset_path, 'data.yaml')
    with open(yaml_path) as f:
        content = f.read()
    
    # Replace kpt_shape
    content = content.replace(
        'kpt_shape:\n- 22\n- 3',
        'kpt_shape:\n- 18\n- 3'
    )
    
    # Add flip_idx if not present
    if 'flip_idx' not in content:
        flip_line = '\nflip_idx: [3, 2, 1, 0, 5, 4, 7, 6, 9, 8, 11, 10, 13, 12, 15, 14, 17, 16]\n'
        content = content.replace(
            'kpt_shape:\n- 18\n- 3',
            'kpt_shape:\n- 18\n- 3' + flip_line
        )
    
    with open(yaml_path, 'w') as f:
        f.write(content)
    
    print("data.yaml updated to kpt_shape [18, 3]")
    print(f"Contents:\n{content}")

if __name__ == '__main__':

    print(f"GPU: {torch.cuda.get_device_name(0)}")

    # Download
    rf = Roboflow(api_key="your_key")
    project = rf.workspace("adityas-workspace-0eca5").project("badmintoncourtdetectionoffical-b3hl9-dloud")
    version = project.version(1)
    dataset = version.download("yolov8")
    print(f"Dataset: {dataset.location}")

    # Filter keypoints
    filter_labels(dataset.location, REMOVE_IDS)

    # Update yaml
    update_yaml(dataset.location)

    # Verify
    data_yaml = os.path.join(dataset.location, 'data.yaml')

    # Train
    model = YOLO('yolo11s-pose.pt')

    print("\nStarting keypoint training...")
    try:
        model.train(
            data=data_yaml,
            epochs=150,
            imgsz=960,
            batch=4,
            device=0,
            cache=False,
            optimizer='AdamW',
            lr0=0.0005,
            weight_decay=0.001,
            workers=2,
            patience=30,
            save=True,
            save_period=5,
            plots=True,
            project=RUNS_DIR,
            name=RUN_NAME,
            exist_ok=True,
            amp=True,
            box=7.5,
            cls=0.5,
            pose=12.0,
            kobj=2.0,
            mosaic=0.0,
            mixup=0.0,
            copy_paste=0.0,
            fliplr=0.5,
            scale=0.1,
            hsv_v=0.2,
            hsv_s=0.1,
        )
        print("\nTraining complete")

    except KeyboardInterrupt:
        print("\nInterrupted")

    except Exception as e:
        print(f"\nCrashed: {e}")

    finally:
        backup_weights(WEIGHTS_DIR, BACKUP_DIR)